In [1]:
from plotly.io import show
from sklearn.model_selection import train_test_split

from skfolio import Population, RiskMeasure
from skfolio.datasets import load_sp500_dataset
from skfolio.moments import ShrunkCovariance
from skfolio.optimization import RiskBudgeting
from skfolio.preprocessing import prices_to_returns
from skfolio.prior import EmpiricalPrior

prices = load_sp500_dataset()

X = prices_to_returns(prices)
X_train, X_test = train_test_split(X, test_size=0.33, shuffle=False)

In [2]:
model = RiskBudgeting(
    risk_measure=RiskMeasure.VARIANCE,
    prior_estimator=EmpiricalPrior(
        covariance_estimator=ShrunkCovariance(shrinkage=0.9)
    ),
    portfolio_params=dict(name="Risk Parity - Covariance Shrinkage"),
)
model.fit(X_train)
model.weights_

array([0.04774564, 0.04370281, 0.04503308, 0.04647756, 0.05284651,
       0.04907501, 0.04853073, 0.05373981, 0.04539461, 0.05360782,
       0.05178479, 0.05137815, 0.04927093, 0.05375797, 0.05112883,
       0.05417652, 0.04754835, 0.04988004, 0.05199129, 0.05292954])

In [3]:
bench = RiskBudgeting(
    risk_measure=RiskMeasure.VARIANCE,
    portfolio_params=dict(name="Risk Parity - Basic"),
)
bench.fit(X_train)
bench.weights_

array([0.04135337, 0.03210817, 0.03372763, 0.03785112, 0.06105294,
       0.04432743, 0.04252252, 0.06593454, 0.03451724, 0.06469309,
       0.05418778, 0.05209434, 0.04535404, 0.06568126, 0.05103923,
       0.06894529, 0.0404657 , 0.04667704, 0.05627166, 0.06119559])

In [4]:
ptf_model_test = model.predict(X_test)
ptf_bench_test = bench.predict(X_test)

In [5]:
population = Population([ptf_model_test, ptf_bench_test])

In [6]:
fig = population.plot_cumulative_returns()
show(fig)

In [7]:
population.summary()

,Risk Parity - Covariance Shrinkage,Risk Parity - Basic
Mean,0.068%,0.065%
Annualized Mean,17.04%,16.40%
Variance,0.011%,0.010%
Annualized Variance,2.84%,2.63%
Semi-Variance,0.0058%,0.0054%
Annualized Semi-Variance,1.47%,1.36%
Standard Deviation,1.06%,1.02%
Annualized Standard Deviation,16.86%,16.22%
Semi-Deviation,0.76%,0.73%
Annualized Semi-Deviation,12.11%,11.66%
